In [ ]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/Kadantte/tennis_atp/master/atp_matches_{}.csv"
years = [2020, 2021, 2022, 2023]
dfs = []

print("Fetching US Open match data...")

for year in years:
    url = base_url.format(year)
    try:
        df = pd.read_csv(url)

        # 1. Filter strictly for US Open matches
        us_open = df[
            df["tourney_name"].str.contains("US Open", case=False, na=False)
        ].copy()

        # 2. Extract or ensure 'tourney_year' exists from 'tourney_date'
        if "tourney_date" in us_open.columns:
            us_open["tourney_year"] = (
                us_open["tourney_date"].astype(str).str[:4].astype(int)
            )
        else:
            us_open["tourney_year"] = year

        dfs.append(us_open)
        print(f"✓ Extracted {len(us_open)} US Open matches for {year}")
    except Exception as e:
        print(f"✗ Error loading {year}: {e}")

# Combine all years into one dataframe
us_open_data = pd.concat(dfs, ignore_index=True)

# Select existing columns safely to preview
cols_to_preview = [
    c
    for c in [
        "tourney_year",
        "round",
        "winner_name",
        "loser_name",
        "winner_rank",
        "loser_rank",
        "score",
    ]
    if c in us_open_data.columns
]

# Save locally to CSV
output_filename = "us_open_matches_2020_2023.csv"
us_open_data.to_csv(output_filename, index=False)

print(
    f"\nDone! Total US Open matches collected: {len(us_open_data)}"
)
print(f"Saved to '{output_filename}'\n")
print(us_open_data[cols_to_preview].head())


# 1. Load the collected US Open data
df = pd.read_csv("us_open_matches_2020_2023.csv")

# 2. Build ML Features
np.random.seed(42)  # For reproducible random A/B assignment
features = []

for _, row in df.iterrows():
    # Randomize whether Winner is assigned to Player A or Player B
    swap = np.random.rand() > 0.5

    pA_rank = row["loser_rank"] if swap else row["winner_rank"]
    pB_rank = row["winner_rank"] if swap else row["loser_rank"]

    pA_age = row["loser_age"] if swap else row["winner_age"]
    pB_age = row["winner_age"] if swap else row["loser_age"]

    pA_ht = row["loser_ht"] if swap else row["winner_ht"]
    pB_ht = row["winner_ht"] if swap else row["loser_ht"]

    # Target label: 1 if Player A won, 0 if Player B won
    target = 0 if swap else 1

    features.append({
        "rank_diff": pB_rank - pA_rank,  # Positive means Player A has a higher (better) rank
        "age_diff": pA_age - pB_age,
        "height_diff": (pA_ht - pB_ht) if pd.notnull(pA_ht) and pd.notnull(pB_ht) else 0,
        "target_player_a_won": target,
    })

ml_df = pd.DataFrame(features).dropna()

# 3. Train/Test Split & Random Forest Baseline
X = ml_df[["rank_diff", "age_diff", "height_diff"]]
y = ml_df["target_player_a_won"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Evaluate Baseline
predictions = model.predict(X_test)
acc = accuracy_score(y_test, predictions)

print(f"Baseline Random Forest Model Accuracy: {acc * 100:.2f}%")

Fetching US Open match data...
✓ Extracted 127 US Open matches for 2020
✓ Extracted 127 US Open matches for 2021
✓ Extracted 127 US Open matches for 2022
✓ Extracted 127 US Open matches for 2023

Done! Total US Open matches collected: 508
Saved to 'us_open_matches_2020_2023.csv'

   tourney_year round          winner_name         loser_name  winner_rank  \
0          2020  R128       Novak Djokovic      Damir Dzumhur          1.0   
1          2020  R128          Kyle Edmund   Alexander Bublik         44.0   
2          2020  R128         Michael Mmoh         Joao Sousa        186.0   
3          2020  R128   Jan Lennard Struff     Pedro Martinez         29.0   
4          2020  R128  Pablo Carreno Busta  Yasutaka Uchiyama         27.0   

   loser_rank                score  
0       109.0          6-1 6-4 6-1  
1        54.0      2-6 7-5 7-5 6-0  
2        68.0      6-2 7-5 2-6 6-1  
3       106.0          6-0 7-5 6-4  
4        86.0  4-6 6-3 1-6 6-3 6-3  
Baseline Random Forest Model

In [ ]:
import pandas as pd

def compute_elo_ratings(all_matches, k_factor=32, default_elo=1500):
    elo_ratings = {}
    elo_history = []

    if "tourney_date" in all_matches.columns:
        all_matches = all_matches.sort_values("tourney_date")

    for _, row in all_matches.iterrows():
        winner, loser = row["winner_name"], row["loser_name"]

        r_winner = elo_ratings.get(winner, default_elo)
        r_loser = elo_ratings.get(loser, default_elo)

        elo_history.append({
            "winner": winner,
            "loser": loser,
            "winner_elo_before": r_winner,
            "loser_elo_before": r_loser,
            "elo_diff": r_winner - r_loser
        })

        exp_winner = 1 / (1 + 10 ** ((r_loser - r_winner) / 400))
        exp_loser = 1 - exp_winner

        elo_ratings[winner] = r_winner + k_factor * (1 - exp_winner)
        elo_ratings[loser] = r_loser + k_factor * (0 - exp_loser)

    return elo_ratings, pd.DataFrame(elo_history)


In [25]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Load the collected US Open data
df = pd.read_csv("us_open_matches_2020_2023.csv")

# 2. Build ML Features
np.random.seed(42)  # For reproducible random A/B assignment
features = []

for _, row in df.iterrows():
    # Randomize whether Winner is assigned to Player A or Player B
    swap = np.random.rand() > 0.5

    pA_rank = row["loser_rank"] if swap else row["winner_rank"]
    pB_rank = row["winner_rank"] if swap else row["loser_rank"]

    pA_age = row["loser_age"] if swap else row["winner_age"]
    pB_age = row["winner_age"] if swap else row["loser_age"]

    pA_ht = row["loser_ht"] if swap else row["winner_ht"]
    pB_ht = row["winner_ht"] if swap else row["loser_ht"]

    # Target label: 1 if Player A won, 0 if Player B won
    target = 0 if swap else 1

    features.append({
        "rank_diff": pB_rank - pA_rank,  # Positive means Player A has a higher (better) rank
        "age_diff": pA_age - pB_age,
        "height_diff": (pA_ht - pB_ht) if pd.notnull(pA_ht) and pd.notnull(pB_ht) else 0,
        "target_player_a_won": target,
    })

ml_df = pd.DataFrame(features).dropna()

# 3. Train/Test Split & Random Forest Baseline
X = ml_df[["rank_diff", "age_diff", "height_diff"]]
y = ml_df["target_player_a_won"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Evaluate Baseline
predictions = model.predict(X_test)
acc = accuracy_score(y_test, predictions)

print(f"Baseline Random Forest Model Accuracy: {acc * 100:.2f}%")

Baseline Random Forest Model Accuracy: 69.31%


In [26]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/Kadantte/tennis_atp/master/atp_matches_{}.csv"
years = [2020, 2021, 2022, 2023]
full_season_dfs = []

print("Fetching full season datasets...")
for year in years:
    try:
        df = pd.read_csv(base_url.format(year))
        # Parse match dates (YYYYMMDD) for chronological ordering
        df["tourney_date"] = pd.to_datetime(
            df["tourney_date"].astype(str), format="%Y%m%d"
        )
        full_season_dfs.append(df)
        print(f"✓ Loaded full season data for {year}")
    except Exception as e:
        print(f"✗ Failed {year}: {e}")

all_matches = pd.concat(full_season_dfs, ignore_index=True)
all_matches = all_matches.sort_values(
    ["tourney_date", "match_num"]
).reset_index(drop=True)
print(f"\nTotal matches loaded: {len(all_matches)}")

Fetching full season datasets...
✓ Loaded full season data for 2020
✓ Loaded full season data for 2021
✓ Loaded full season data for 2022
✓ Loaded full season data for 2023

Total matches loaded: 10098


In [27]:
import numpy as np


def get_player_stats_before_date(matches_df, player_name, match_date, N=10):
    """Calculates rolling form and surface-specific stats for a player

    STRICTLY BEFORE the match_date (preventing data leakage).
    """
    # Filter for matches involving this player before the current match date
    past = matches_df[
        (matches_df["tourney_date"] < match_date)
        & (
            (matches_df["winner_name"] == player_name)
            | (matches_df["loser_name"] == player_name)
        )
    ]

    if len(past) == 0:
        return {"rolling_win_pct": 0.5, "hard_win_pct": 0.5}

    # 1. Overall Rolling Win % (Last N Matches)
    recent = past.tail(N)
    recent_wins = (recent["winner_name"] == player_name).sum()
    rolling_win_pct = recent_wins / len(recent)

    # 2. Hard Court Specific Win %
    hard_matches = past[past["surface"] == "Hard"]
    if len(hard_matches) > 0:
        hard_wins = (hard_matches["winner_name"] == player_name).sum()
        hard_win_pct = hard_wins / len(hard_matches)
    else:
        hard_win_pct = 0.5

    return {"rolling_win_pct": rolling_win_pct, "hard_win_pct": hard_win_pct}


print("Building advanced features...")
us_open_matches = all_matches[
    all_matches["tourney_name"].str.contains("US Open", case=False, na=False)
].copy()

np.random.seed(42)
ml_rows = []

for idx, row in us_open_matches.iterrows():
    match_date = row["tourney_date"]
    winner = row["winner_name"]
    loser = row["loser_name"]

    # Randomize Player A / Player B target
    swap = np.random.rand() > 0.5
    pA_name = loser if swap else winner
    pB_name = winner if swap else loser
    target = 0 if swap else 1

    # Ranks & Points
    pA_rank = row["loser_rank"] if swap else row["winner_rank"]
    pB_rank = row["winner_rank"] if swap else row["loser_rank"]

    # Fetch prior rolling stats
    pA_stats = get_player_stats_before_date(all_matches, pA_name, match_date)
    pB_stats = get_player_stats_before_date(all_matches, pB_name, match_date)

    ml_rows.append({
        "rank_diff": (pB_rank - pA_rank)
        if pd.notnull(pA_rank) and pd.notnull(pB_rank)
        else 0,
        "form_diff_last10": pA_stats["rolling_win_pct"]
        - pB_stats["rolling_win_pct"],
        "hard_win_pct_diff": pA_stats["hard_win_pct"]
        - pB_stats["hard_win_pct"],
        "target": target,
    })

ml_df = pd.DataFrame(ml_rows)
print(f"✓ Engineered features for {len(ml_df)} US Open matches!")
print(ml_df.head())

Building advanced features...
✓ Engineered features for 508 US Open matches!
   rank_diff  form_diff_last10  hard_win_pct_diff  target
0      108.0          0.714286           0.714286       1
1      -10.0         -0.100000          -0.087302       0
2      118.0         -0.166667          -0.166667       0
3      -77.0         -0.200000          -0.125000       0
4       59.0          0.333333           0.371795       1


In [28]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

# ---------------------------------------------------------
# STEP 1: FETCH HISTORICAL & RECENT ATP MATCH DATA
# ---------------------------------------------------------
print("1. Fetching match datasets dynamically from remote source...")
base_url = "https://raw.githubusercontent.com/Kadantte/tennis_atp/master/atp_matches_{}.csv"

# Fetch recent available ATP match seasons
years = [2020, 2021, 2022, 2023]
dfs = []

for year in years:
    try:
        url = base_url.format(year)
        df = pd.read_csv(url)
        df["tourney_date"] = pd.to_datetime(
            df["tourney_date"].astype(str), format="%Y%m%d"
        )
        dfs.append(df)
        print(f"✓ Fetched match records for {year}")
    except Exception as e:
        print(f"✗ Failed to fetch {year}: {e}")

if not dfs:
    raise RuntimeError("Failed to fetch match datasets.")

# Combine into chronologically sorted master dataframe
all_matches = pd.concat(dfs, ignore_index=True).sort_values("tourney_date")


# ---------------------------------------------------------
# STEP 2: DYNAMIC STAT CALCULATOR FUNCTION
# ---------------------------------------------------------
def get_player_stats(
    matches_df, player_name, match_date=None, N=10, hard_only=False
):
    """Dynamically computes a player's rolling form and surface win %

    directly from the fetched match logs.
    """
    if match_date is not None:
        past = matches_df[matches_df["tourney_date"] < match_date]
    else:
        past = matches_df

    # Filter for matches involving this player
    player_past = past[
        (past["winner_name"] == player_name)
        | (past["loser_name"] == player_name)
    ]

    if len(player_past) == 0:
        return {"rolling_win_pct": 0.5, "hard_win_pct": 0.5}

    # 1. Rolling Win % (Last N Matches)
    recent = player_past.tail(N)
    rolling_win_pct = (recent["winner_name"] == player_name).sum() / len(recent)

    # 2. Hard Court Specific Win %
    hard_matches = player_past[player_past["surface"] == "Hard"]
    if len(hard_matches) > 0:
        hard_win_pct = (hard_matches["winner_name"] == player_name).sum() / len(
            hard_matches
        )
    else:
        hard_win_pct = 0.5

    return {"rolling_win_pct": rolling_win_pct, "hard_win_pct": hard_win_pct}


# ---------------------------------------------------------
# STEP 3: DYNAMIC FEATURE ENGINEERING & MODEL TRAINING
# ---------------------------------------------------------
print("\n2. Dynamically engineering features (Rank, Form, Hard Court Win %)...")
us_open_historical = all_matches[
    all_matches["tourney_name"].str.contains("US Open", case=False, na=False)
].copy()

np.random.seed(42)
ml_rows = []

for _, row in us_open_historical.iterrows():
    date = row["tourney_date"]
    winner, loser = row["winner_name"], row["loser_name"]

    # Randomize Player A / Player B target
    swap = np.random.rand() > 0.5
    pA, pB = (loser, winner) if swap else (winner, loser)
    target = 0 if swap else 1

    pA_rank = row["loser_rank"] if swap else row["winner_rank"]
    pB_rank = row["winner_rank"] if swap else row["loser_rank"]

    pA_stats = get_player_stats(all_matches, pA, match_date=date)
    pB_stats = get_player_stats(all_matches, pB, match_date=date)

    ml_rows.append({
        "rank_diff": (pB_rank - pA_rank)
        if pd.notnull(pA_rank) and pd.notnull(pB_rank)
        else 0,
        "form_diff": pA_stats["rolling_win_pct"] - pB_stats["rolling_win_pct"],
        "hard_diff": pA_stats["hard_win_pct"] - pB_stats["hard_win_pct"],
        "target": target,
    })

ml_df = pd.DataFrame(ml_rows)

print("3. Training HistGradientBoostingClassifier model...")
X = ml_df[["rank_diff", "form_diff", "hard_diff"]]
y = ml_df["target"]

model = HistGradientBoostingClassifier(random_state=42)
model.fit(X, y)
print("✓ Model trained on dynamically constructed features!")

# ---------------------------------------------------------
# STEP 4: DYNAMICALLY EXTRACT 2026 TOP PLAYERS & RANKINGS
# ---------------------------------------------------------
print("\n4. Dynamically extracting top players & current ranks from data...")

# Get the most recent match for each player to find their latest rank & stats
latest_matches = all_matches.sort_values("tourney_date").copy()

# Extract players with their most recent recorded rank
winners = latest_matches[["winner_name", "winner_rank"]].rename(
    columns={"winner_name": "name", "winner_rank": "rank"}
)
losers = latest_matches[["loser_name", "loser_rank"]].rename(
    columns={"loser_name": "name", "loser_rank": "rank"}
)
all_player_ranks = (
    pd.concat([winners, losers])
    .dropna()
    .groupby("name")
    .last()
    .reset_index()
)

# Select top 8 ranked active players from the fetched dataset
top_players_df = all_player_ranks.sort_values("rank").head(8)

# Dynamically build top player profiles from fetched dataset
players_2026 = []
for _, row in top_players_df.iterrows():
    p_name = row["name"]
    p_rank = int(row["rank"])

    # Compute current stats dynamically from the dataset
    stats = get_player_stats(all_matches, p_name)

    players_2026.append({
        "name": p_name,
        "rank": p_rank,
        "form": stats["rolling_win_pct"],
        "hard_pct": stats["hard_win_pct"],
    })

print("✓ Dynamically constructed player roster:")
for p in players_2026:
    print(
        f"  - Rank #{p['rank']}: {p['name']} | Form: {p['form'] * 100:.1f}% | Hard Court Win %: {p['hard_pct'] * 100:.1f}%"
    )

# ---------------------------------------------------------
# STEP 5: SIMULATE 2026 US OPEN
# ---------------------------------------------------------
print("\n" + "=" * 50)
print(" 🎾 SIMULATING 2026 US OPEN TOURNAMENT 🎾")
print("=" * 50)


def predict_match(pA, pB):
    rank_diff = pB["rank"] - pA["rank"]
    form_diff = pA["form"] - pB["form"]
    hard_diff = pA["hard_pct"] - pB["hard_pct"]

    features = pd.DataFrame(
        [[rank_diff, form_diff, hard_diff]],
        columns=["rank_diff", "form_diff", "hard_diff"],
    )
    prob_pA_win = model.predict_proba(features)[0][1]

    winner = pA if prob_pA_win >= 0.5 else pB
    win_prob = prob_pA_win if prob_pA_win >= 0.5 else (1 - prob_pA_win)
    return winner, win_prob


# Simulate Quarterfinals
print("\n--- QUARTERFINALS ---")
qf_matchups = [
    (players_2026[0], players_2026[7]),
    (players_2026[3], players_2026[4]),
    (players_2026[2], players_2026[5]),
    (players_2026[1], players_2026[6]),
]

sf_players = []
for p1, p2 in qf_matchups:
    winner, win_prob = predict_match(p1, p2)
    sf_players.append(winner)
    print(
        f"{p1['name']} (# {p1['rank']}) vs {p2['name']} (# {p2['rank']}) ➔ Winner: {winner['name']} ({win_prob * 100:.1f}% win probability)"
    )

# Simulate Semifinals
print("\n--- SEMIFINALS ---")
sf_matchups = [(sf_players[0], sf_players[1]), (sf_players[2], sf_players[3])]

finalists = []
for p1, p2 in sf_matchups:
    winner, win_prob = predict_match(p1, p2)
    finalists.append(winner)
    print(
        f"{p1['name']} vs {p2['name']} ➔ Winner: {winner['name']} ({win_prob * 100:.1f}% win probability)"
    )

# Simulate Championship Final
print("\n--- CHAMPIONSHIP FINAL ---")
champion, win_prob = predict_match(finalists[0], finalists[1])
print(
    f"\n🏆 CHAMPIONSHIP MATCH: {finalists[0]['name']} vs {finalists[1]['name']}"
)
print(
    f"🏆 PREDICTED 2026 US OPEN CHAMPION: {champion['name']} ({win_prob * 100:.1f}% confidence)"
)

1. Fetching match datasets dynamically from remote source...
✓ Fetched match records for 2020
✓ Fetched match records for 2021
✓ Fetched match records for 2022
✓ Fetched match records for 2023

2. Dynamically engineering features (Rank, Form, Hard Court Win %)...
3. Training HistGradientBoostingClassifier model...
✓ Model trained on dynamically constructed features!

4. Dynamically extracting top players & current ranks from data...
✓ Dynamically constructed player roster:
  - Rank #1: Novak Djokovic | Form: 80.0% | Hard Court Win %: 89.6%
  - Rank #2: Carlos Alcaraz | Form: 60.0% | Hard Court Win %: 73.5%
  - Rank #2: Rafael Nadal | Form: 40.0% | Hard Court Win %: 74.2%
  - Rank #3: Daniil Medvedev | Form: 60.0% | Hard Court Win %: 79.5%
  - Rank #4: Jannik Sinner | Form: 80.0% | Hard Court Win %: 74.9%
  - Rank #5: Andrey Rublev | Form: 50.0% | Hard Court Win %: 71.2%
  - Rank #6: Stefanos Tsitsipas | Form: 60.0% | Hard Court Win %: 67.6%
  - Rank #7: Alexander Zverev | Form: 60.0% |

In [29]:
import math
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import HistGradientBoostingClassifier


# =====================================================================
# MODULE 1: RECENCY-WEIGHTED ELO ENGINE WITH TIME DECAY
# =====================================================================
class RecencyWeightedElo:

    def __init__(self, base_elo=1500, default_k=32, half_life_days=365):
        self.elo_table = {}
        self.base_elo = base_elo
        self.default_k = default_k
        self.half_life_days = half_life_days

    def get_elo(self, player_name):
        return self.elo_table.get(player_name, self.base_elo)

    def calculate_expected_score(self, rating_a, rating_b):
        return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

    def process_matches(
        self, matches_df, reference_date=pd.Timestamp("2026-08-01")
    ):
        """Processes historical matches in chronological order applying time-decay weighting."""
        sorted_matches = matches_df.sort_values("tourney_date")

        for _, row in sorted_matches.iterrows():
            w = row["winner_name"]
            l = row["loser_name"]
            match_date = row["tourney_date"]

            r_w = self.get_elo(w)
            r_l = self.get_elo(l)

            e_w = self.calculate_expected_score(r_w, r_l)
            e_l = 1 - e_w

            # Time-decay factor: older matches carry less weight
            days_old = (reference_date - match_date).days
            decay = 0.5 ** (max(0, days_old) / self.half_life_days)
            k_time = self.default_k * decay

            self.elo_table[w] = r_w + k_time * (1 - e_w)
            self.elo_table[l] = r_l + k_time * (0 - e_l)


# =====================================================================
# MODULE 2: LIVE WEATHER API INTEGRATION & IMPACT ADJUSTMENT
# =====================================================================
def get_us_open_weather(date_str="2025-09-01"):
    """Fetches weather for Flushing Meadows, NY via Open-Meteo API."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 40.7500,  # USTA Billie Jean King National Tennis Center
        "longitude": -73.8472,
        "start_date": date_str,
        "end_date": date_str,
        "hourly": ["temperature_2m", "wind_speed_10m"],
    }
    try:
        res = requests.get(url, params=params, timeout=5).json()
        temp_max = max(res["hourly"]["temperature_2m"])  # °C
        wind_max = max(res["hourly"]["wind_speed_10m"])  # km/h
        return {"max_temp_c": temp_max, "max_wind_kmh": wind_max}
    except Exception:
        # Mild outdoor baseline fallback
        return {"max_temp_c": 28.0, "max_wind_kmh": 15.0}


def apply_weather_adjustment(elo, ace_rate, stamina, weather):
    """Adjusts player's effective Elo based on ambient court weather."""
    adj_elo = elo

    # 1. Extreme Heat (>30°C / 86°F): Boosts high stamina, punishes low stamina
    if weather["max_temp_c"] > 30.0:
        adj_elo += (stamina - 0.5) * 40.0

    # 2. High Wind (>20 km/h): Disrupts toss mechanics for high-ace servers
    if weather["max_wind_kmh"] > 20.0 and ace_rate > 0.8:
        adj_elo -= 25.0

    return adj_elo


# =====================================================================
# MODULE 3: DATA INGESTION & PIPELINE ASSEMBLY
# =====================================================================
print("1. Ingesting remote ATP datasets...")
base_url = "https://raw.githubusercontent.com/Kadantte/tennis_atp/master/atp_matches_{}.csv"
years = [2020, 2021, 2022, 2023]
dfs = []

for year in years:
    try:
        df = pd.read_csv(base_url.format(year))
        df["tourney_date"] = pd.to_datetime(
            df["tourney_date"].astype(str), format="%Y%m%d"
        )
        dfs.append(df)
        print(f"  ✓ Processed {year} season")
    except Exception as e:
        print(f"  ✗ Failed year {year}: {e}")

all_matches = pd.concat(dfs, ignore_index=True).sort_values("tourney_date")

print("2. Computing Recency-Weighted Elo Ratings...")
elo_engine = RecencyWeightedElo(half_life_days=365)
elo_engine.process_matches(all_matches)

print("3. Fetching Matchday Weather Data...")
matchday_weather = get_us_open_weather()
print(
    f"  ✓ Weather conditions: Temp = {matchday_weather['max_temp_c']}°C, Wind = {matchday_weather['max_wind_kmh']} km/h"
)


# Helper: Form & Hard Court Stats
def get_player_form(matches_df, name, N=10):
    past = matches_df[
        (matches_df["winner_name"] == name)
        | (matches_df["loser_name"] == name)
    ].tail(N)
    if len(past) == 0:
        return 0.50, 0.50
    form = (past["winner_name"] == name).sum() / len(past)

    hard = matches_df[
        ((matches_df["winner_name"] == name) | (matches_df["loser_name"] == name))
        & (matches_df["surface"] == "Hard")
    ]
    hard_pct = (
        (hard["winner_name"] == name).sum() / len(hard) if len(hard) > 0 else 0.50
    )
    return form, hard_pct


# =====================================================================
# MODULE 4: FEATURE ENGINEERING & MODEL TRAINING
# =====================================================================
print("\n4. Engineering Machine Learning Feature Matrix...")
us_open_matches = all_matches[
    all_matches["tourney_name"].str.contains("US Open", case=False, na=False)
].copy()

np.random.seed(42)
ml_rows = []

for _, row in us_open_matches.iterrows():
    winner, loser = row["winner_name"], row["loser_name"]

    # Randomize Player A/B to prevent classification bias
    swap = np.random.rand() > 0.5
    pA, pB = (loser, winner) if swap else (winner, loser)
    target = 0 if swap else 1

    pA_elo = elo_engine.get_elo(pA)
    pB_elo = elo_engine.get_elo(pB)

    pA_form, pA_hard = get_player_form(all_matches, pA)
    pB_form, pB_hard = get_player_form(all_matches, pB)

    ml_rows.append({
        "elo_diff": pA_elo - pB_elo,
        "form_diff": pA_form - pB_form,
        "hard_diff": pA_hard - pB_hard,
        "target": target,
    })

ml_df = pd.DataFrame(ml_rows)

print("5. Training HistGradientBoosting Classifier...")
X = ml_df[["elo_diff", "form_diff", "hard_diff"]]
y = ml_df["target"]

model = HistGradientBoostingClassifier(random_state=42)
model.fit(X, y)
print("✓ Model successfully trained!")

# =====================================================================
# MODULE 5: 2026 US OPEN TOURNAMENT SIMULATION
# =====================================================================
print("\n" + "=" * 60)
print(" 🎾 SIMULATING THE 2026 US OPEN (WEATHER & ELO ADJUSTED) 🎾")
print("=" * 60)

# Build Top Player Profiles directly from recent data
recent_ranks = (
    pd.concat([
        all_matches[["winner_name", "winner_rank"]].rename(
            columns={"winner_name": "name", "winner_rank": "rank"}
        ),
        all_matches[["loser_name", "loser_rank"]].rename(
            columns={"loser_name": "name", "loser_rank": "rank"}
        ),
    ])
    .dropna()
    .groupby("name")
    .last()
    .reset_index()
    .sort_values("rank")
    .head(8)
)

roster = []
for _, row in recent_ranks.iterrows():
    name = row["name"]
    rank = int(row["rank"])
    raw_elo = elo_engine.get_elo(name)
    form, hard_pct = get_player_form(all_matches, name)

    # Simulated stamina/ace metrics for weather adjustments
    stamina = 0.85 if rank <= 3 else 0.65
    ace_rate = 0.90 if "Medvedev" in name or "Zverev" in name else 0.50

    adj_elo = apply_weather_adjustment(
        raw_elo, ace_rate, stamina, matchday_weather
    )

    roster.append({
        "name": name,
        "rank": rank,
        "raw_elo": raw_elo,
        "adj_elo": adj_elo,
        "form": form,
        "hard_pct": hard_pct,
    })

print("\n--- QUALIFIED TOP 8 SEEDS ---")
for p in roster:
    print(
        f"Seed #{p['rank']} {p['name']:<20} | Base Elo: {p['raw_elo']:.1f} ➔ Weather Elo: {p['adj_elo']:.1f}"
    )


def predict_matchup(pA, pB):
    elo_diff = pA["adj_elo"] - pB["adj_elo"]
    form_diff = pA["form"] - pB["form"]
    hard_diff = pA["hard_pct"] - pB["hard_pct"]

    feats = pd.DataFrame(
        [[elo_diff, form_diff, hard_diff]],
        columns=["elo_diff", "form_diff", "hard_diff"],
    )
    pA_win_prob = model.predict_proba(feats)[0][1]

    winner = pA if pA_win_prob >= 0.5 else pB
    win_prob = pA_win_prob if pA_win_prob >= 0.5 else (1 - pA_win_prob)
    return winner, win_prob


# Simulate Tournament Bracket
print("\n--- QUARTERFINALS ---")
qf_pairs = [
    (roster[0], roster[7]),
    (roster[3], roster[4]),
    (roster[2], roster[5]),
    (roster[1], roster[6]),
]
sf_roster = []

for p1, p2 in qf_pairs:
    w, prob = predict_matchup(p1, p2)
    sf_roster.append(w)
    print(
        f"{p1['name']} vs {p2['name']:<18} ➔ Winner: {w['name']} ({prob * 100:.1f}% confidence)"
    )

print("\n--- SEMIFINALS ---")
sf_pairs = [(sf_roster[0], sf_roster[1]), (sf_roster[2], sf_roster[3])]
finalists = []

for p1, p2 in sf_pairs:
    w, prob = predict_matchup(p1, p2)
    finalists.append(w)
    print(
        f"{p1['name']} vs {p2['name']:<18} ➔ Winner: {w['name']} ({prob * 100:.1f}% confidence)"
    )

print("\n--- CHAMPIONSHIP FINAL ---")
champion, prob = predict_matchup(finalists[0], finalists[1])
print(f"🏆 MATCHUP: {finalists[0]['name']} vs {finalists[1]['name']}")
print(
    f"🏆 PREDICTED CHAMPION: {champion['name']} ({prob * 100:.1f}% win probability)"
)

1. Ingesting remote ATP datasets...
  ✓ Processed 2020 season
  ✓ Processed 2021 season
  ✓ Processed 2022 season
  ✓ Processed 2023 season
2. Computing Recency-Weighted Elo Ratings...
3. Fetching Matchday Weather Data...
  ✓ Weather conditions: Temp = 25.7°C, Wind = 15.4 km/h

4. Engineering Machine Learning Feature Matrix...
5. Training HistGradientBoosting Classifier...
✓ Model successfully trained!

 🎾 SIMULATING THE 2026 US OPEN (WEATHER & ELO ADJUSTED) 🎾

--- QUALIFIED TOP 8 SEEDS ---
Seed #1 Novak Djokovic       | Base Elo: 1632.4 ➔ Weather Elo: 1632.4
Seed #2 Carlos Alcaraz       | Base Elo: 1612.2 ➔ Weather Elo: 1612.2
Seed #2 Rafael Nadal         | Base Elo: 1530.9 ➔ Weather Elo: 1530.9
Seed #3 Daniil Medvedev      | Base Elo: 1605.5 ➔ Weather Elo: 1605.5
Seed #4 Jannik Sinner        | Base Elo: 1610.3 ➔ Weather Elo: 1610.3
Seed #5 Andrey Rublev        | Base Elo: 1580.5 ➔ Weather Elo: 1580.5
Seed #6 Stefanos Tsitsipas   | Base Elo: 1570.4 ➔ Weather Elo: 1570.4
Seed #7 Alexan

In [ ]:
import math
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import HistGradientBoostingClassifier


# =====================================================================
# MODULE 1: RECENCY-WEIGHTED ELO ENGINE WITH TIME DECAY
# =====================================================================
class RecencyWeightedElo:

    def __init__(self, base_elo=1500, default_k=32, half_life_days=365):
        self.elo_table = {}
        self.base_elo = base_elo
        self.default_k = default_k
        self.half_life_days = half_life_days

    def get_elo(self, player_name):
        return self.elo_table.get(player_name, self.base_elo)

    def calculate_expected_score(self, rating_a, rating_b):
        return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

    def process_matches(
        self, matches_df, reference_date=pd.Timestamp("2026-08-01")
    ):
        """Processes historical matches in chronological order applying time-decay weighting."""
        sorted_matches = matches_df.sort_values("tourney_date")

        for _, row in sorted_matches.iterrows():
            w = row["winner_name"]
            l = row["loser_name"]
            match_date = row["tourney_date"]

            r_w = self.get_elo(w)
            r_l = self.get_elo(l)

            e_w = self.calculate_expected_score(r_w, r_l)
            e_l = 1 - e_w

            # Time-decay factor: older matches carry less weight
            days_old = (reference_date - match_date).days
            decay = 0.5 ** (max(0, days_old) / self.half_life_days)
            k_time = self.default_k * decay

            self.elo_table[w] = r_w + k_time * (1 - e_w)
            self.elo_table[l] = r_l + k_time * (0 - e_l)


# =====================================================================
# MODULE 2: LIVE WEATHER API INTEGRATION & IMPACT ADJUSTMENT
# =====================================================================
def get_us_open_weather(date_str="2025-09-01"):
    """Fetches weather for Flushing Meadows, NY via Open-Meteo API with safe fallbacks."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 40.7500,  # USTA Billie Jean King National Tennis Center
        "longitude": -73.8472,
        "start_date": date_str,
        "end_date": date_str,
        "hourly": ["temperature_2m", "wind_speed_10m"],
    }
    try:
        res = requests.get(url, params=params, timeout=5).json()
        if "hourly" in res:
            temp_max = max(res["hourly"]["temperature_2m"])  # °C
            wind_max = max(res["hourly"]["wind_speed_10m"])  # km/h
            return {"max_temp_c": temp_max, "max_wind_kmh": wind_max}
    except Exception:
        pass

    # Standard Flushing Meadows late-summer baseline fallback
    return {"max_temp_c": 28.0, "max_wind_kmh": 15.0}


def apply_weather_adjustment(elo, ace_rate, stamina, weather):
    """Adjusts player's effective Elo based on ambient court weather."""
    adj_elo = elo

    # 1. Extreme Heat (>30°C / 86°F): Boosts high stamina, punishes low stamina
    if weather["max_temp_c"] > 30.0:
        adj_elo += (stamina - 0.5) * 40.0

    # 2. High Wind (>20 km/h): Disrupts toss mechanics for high-ace servers
    if weather["max_wind_kmh"] > 20.0 and ace_rate > 0.8:
        adj_elo -= 25.0

    return adj_elo


# =====================================================================
# MODULE 3: DATA INGESTION & PIPELINE ASSEMBLY
# =====================================================================
print("1. Ingesting remote ATP datasets...")
base_url = "https://raw.githubusercontent.com/Kadantte/tennis_atp/master/atp_matches_{}.csv"
years = [2020, 2021, 2022, 2023]
dfs = []

for year in years:
    try:
        df = pd.read_csv(base_url.format(year))
        df["tourney_date"] = pd.to_datetime(
            df["tourney_date"].astype(str), format="%Y%m%d"
        )
        dfs.append(df)
        print(f"  ✓ Processed {year} season")
    except Exception as e:
        print(f"  ✗ Failed year {year}: {e}")

all_matches = pd.concat(dfs, ignore_index=True).sort_values("tourney_date")

print("2. Computing Recency-Weighted Elo Ratings...")
elo_engine = RecencyWeightedElo(half_life_days=365)
elo_engine.process_matches(all_matches)

print("3. Fetching Matchday Weather Data...")
matchday_weather = get_us_open_weather()
print(
    f"  ✓ Weather conditions: Temp = {matchday_weather['max_temp_c']}°C, Wind = {matchday_weather['max_wind_kmh']} km/h"
)


# Helper: Form & Hard Court Stats
def get_player_form(matches_df, name, N=10):
    past = matches_df[
        (matches_df["winner_name"] == name)
        | (matches_df["loser_name"] == name)
    ].tail(N)
    if len(past) == 0:
        return 0.50, 0.50
    form = (past["winner_name"] == name).sum() / len(past)

    hard = matches_df[
        ((matches_df["winner_name"] == name) | (matches_df["loser_name"] == name))
        & (matches_df["surface"] == "Hard")
    ]
    hard_pct = (
        (hard["winner_name"] == name).sum() / len(hard) if len(hard) > 0 else 0.50
    )
    return form, hard_pct


# =====================================================================
# MODULE 4: FEATURE ENGINEERING & MODEL TRAINING
# =====================================================================
print("\n4. Engineering Machine Learning Feature Matrix (with Head-to-Head)...")
us_open_matches = all_matches[
    all_matches["tourney_name"].str.contains("US Open", case=False, na=False)
].copy()

h2h_tracker = {}  # Tracks past matchup history dynamically
np.random.seed(42)
ml_rows = []

for _, row in us_open_matches.iterrows():
    winner, loser = row["winner_name"], row["loser_name"]

    # Randomize Player A/B to prevent target bias
    swap = np.random.rand() > 0.5
    pA, pB = (loser, winner) if swap else (winner, loser)
    target = 0 if swap else 1

    pA_elo = elo_engine.get_elo(pA)
    pB_elo = elo_engine.get_elo(pB)

    pA_form, pA_hard = get_player_form(all_matches, pA)
    pB_form, pB_hard = get_player_form(all_matches, pB)

    # Compute dynamic head-to-head differential
    pair_key = tuple(sorted([pA, pB]))
    h2h_data = h2h_tracker.get(pair_key, {pA: 0, pB: 0})
    h2h_diff = h2h_data.get(pA, 0) - h2h_data.get(pB, 0)

    # Update H2H tracker after extracting feature
    if pair_key not in h2h_tracker:
        h2h_tracker[pair_key] = {winner: 1, loser: 0}
    else:
        h2h_tracker[pair_key][winner] = h2h_tracker[pair_key].get(winner, 0) + 1

    ml_rows.append({
        "elo_diff": pA_elo - pB_elo,
        "form_diff": pA_form - pB_form,
        "hard_diff": pA_hard - pB_hard,
        "h2h_diff": h2h_diff,
        "target": target,
    })

ml_df = pd.DataFrame(ml_rows)

print("5. Training HistGradientBoosting Classifier...")
X = ml_df[["elo_diff", "form_diff", "hard_diff", "h2h_diff"]]
y = ml_df["target"]

model = HistGradientBoostingClassifier(random_state=42)
model.fit(X, y)
print("✓ Model successfully trained!")

# =====================================================================
# MODULE 5: 2026 US OPEN TOURNAMENT SIMULATION
# =====================================================================
print("\n" + "=" * 60)
print(" 🎾 SIMULATING THE 2026 US OPEN (WEATHER & ELO ADJUSTED) 🎾")
print("=" * 60)

# Build Top Player Profiles directly from recent data
recent_ranks = (
    pd.concat([
        all_matches[["winner_name", "winner_rank"]].rename(
            columns={"winner_name": "name", "winner_rank": "rank"}
        ),
        all_matches[["loser_name", "loser_rank"]].rename(
            columns={"loser_name": "name", "loser_rank": "rank"}
        ),
    ])
    .dropna()
    .groupby("name")
    .last()
    .reset_index()
    .sort_values("rank")
    .head(8)
)

roster = []
for _, row in recent_ranks.iterrows():
    name = row["name"]
    rank = int(row["rank"])
    raw_elo = elo_engine.get_elo(name)
    form, hard_pct = get_player_form(all_matches, name)

    # Simulated stamina/ace metrics for weather adjustments
    stamina = 0.85 if rank <= 3 else 0.65
    ace_rate = 0.90 if "Medvedev" in name or "Zverev" in name else 0.50

    adj_elo = apply_weather_adjustment(
        raw_elo, ace_rate, stamina, matchday_weather
    )

    roster.append({
        "name": name,
        "rank": rank,
        "raw_elo": raw_elo,
        "adj_elo": adj_elo,
        "form": form,
        "hard_pct": hard_pct,
    })

print("\n--- QUALIFIED TOP 8 SEEDS ---")
for p in roster:
    print(
        f"Seed #{p['rank']} {p['name']:<20} | Base Elo: {p['raw_elo']:.1f} ➔ Weather Elo: {p['adj_elo']:.1f}"
    )


def predict_matchup(pA, pB):
    elo_diff = pA["adj_elo"] - pB["adj_elo"]
    form_diff = pA["form"] - pB["form"]
    hard_diff = pA["hard_pct"] - pB["hard_pct"]

    # Retrieve Head-to-Head differential
    pair_key = tuple(sorted([pA["name"], pB["name"]]))
    h2h_data = h2h_tracker.get(pair_key, {pA["name"]: 0, pB["name"]: 0})
    h2h_diff = h2h_data.get(pA["name"], 0) - h2h_data.get(pB["name"], 0)

    feats = pd.DataFrame(
        [[elo_diff, form_diff, hard_diff, h2h_diff]],
        columns=["elo_diff", "form_diff", "hard_diff", "h2h_diff"],
    )
    pA_win_prob = model.predict_proba(feats)[0][1]

    winner = pA if pA_win_prob >= 0.5 else pB
    win_prob = pA_win_prob if pA_win_prob >= 0.5 else (1 - pA_win_prob)
    return winner, win_prob


# Simulate Tournament Bracket
print("\n--- QUARTERFINALS ---")
qf_pairs = [
    (roster[0], roster[7]),
    (roster[3], roster[4]),
    (roster[2], roster[5]),
    (roster[1], roster[6]),
]
sf_roster = []

for p1, p2 in qf_pairs:
    w, prob = predict_matchup(p1, p2)
    sf_roster.append(w)
    print(
        f"{p1['name']} vs {p2['name']:<18} ➔ Winner: {w['name']} ({prob * 100:.1f}% confidence)"
    )

print("\n--- SEMIFINALS ---")
sf_pairs = [(sf_roster[0], sf_roster[1]), (sf_roster[2], sf_roster[3])]
finalists = []

for p1, p2 in sf_pairs:
    w, prob = predict_matchup(p1, p2)
    finalists.append(w)
    print(
        f"{p1['name']} vs {p2['name']:<18} ➔ Winner: {w['name']} ({prob * 100:.1f}% confidence)"
    )

print("\n--- CHAMPIONSHIP FINAL ---")
champion, prob = predict_matchup(finalists[0], finalists[1])
print(f"🏆 MATCHUP: {finalists[0]['name']} vs {finalists[1]['name']}")
print(
    f"🏆 PREDICTED CHAMPION: {champion['name']} ({prob * 100:.1f}% win probability)"
)

1. Ingesting remote ATP datasets...
  ✓ Processed 2020 season
  ✓ Processed 2021 season
  ✓ Processed 2022 season
  ✓ Processed 2023 season
2. Computing Recency-Weighted Elo Ratings...
3. Fetching Matchday Weather Data...
  ✓ Weather conditions: Temp = 25.7°C, Wind = 15.4 km/h

4. Engineering Machine Learning Feature Matrix (with Head-to-Head)...
5. Training HistGradientBoosting Classifier...
✓ Model successfully trained!

 🎾 SIMULATING THE 2026 US OPEN (WEATHER & ELO ADJUSTED) 🎾

--- QUALIFIED TOP 18 SEEDS ---
Seed #1 Novak Djokovic       | Base Elo: 1632.4 ➔ Weather Elo: 1632.4
Seed #2 Carlos Alcaraz       | Base Elo: 1612.2 ➔ Weather Elo: 1612.2
Seed #2 Rafael Nadal         | Base Elo: 1530.9 ➔ Weather Elo: 1530.9
Seed #3 Daniil Medvedev      | Base Elo: 1605.5 ➔ Weather Elo: 1605.5
Seed #4 Jannik Sinner        | Base Elo: 1610.3 ➔ Weather Elo: 1610.3
Seed #5 Andrey Rublev        | Base Elo: 1580.5 ➔ Weather Elo: 1580.5
Seed #6 Stefanos Tsitsipas   | Base Elo: 1570.4 ➔ Weather Elo: 